In [0]:
%python
from pyspark.sql import functions as F  # ← add this

outbound = spark.sql("""
    SELECT destination AS country, SUM(flight_count) AS outbound
    FROM flights_clean WHERE origin = 'United States'
    GROUP BY destination
""")

inbound = spark.sql("""
    SELECT origin AS country, SUM(flight_count) AS inbound
    FROM flights_clean WHERE destination = 'United States'
    GROUP BY origin
""")

net_flow = outbound.join(inbound, "country", "outer").fillna(0) \
    .withColumn("net_flow", F.col("outbound") - F.col("inbound")) \
    .orderBy(F.desc("net_flow"))

display(net_flow)


country,outbound,inbound,net_flow
Brazil,853,619,234
South Korea,1048,827,221
Germany,1468,1336,132
Netherlands,776,660,116
Guatemala,397,318,79
Australia,329,258,71
Paraguay,60,0,60
United Kingdom,2025,1970,55
El Salvador,561,508,53
Japan,1548,1496,52


Databricks visualization. Run in Databricks to view.

In [0]:
%python
display(spark.sql("""
    SELECT a.origin, a.destination,
           a.flight_count AS forward,
           b.flight_count AS reverse,
           ABS(a.flight_count - b.flight_count) AS imbalance
    FROM flights_clean a
    JOIN flights_clean b
      ON a.origin = b.destination AND a.destination = b.origin
    ORDER BY imbalance DESC
"""))  # ← wrap in display()

origin,destination,forward,reverse,imbalance
United States,Brazil,853,619,234
Brazil,United States,619,853,234
South Korea,United States,827,1048,221
United States,South Korea,1048,827,221
United States,China,772,920,148
China,United States,920,772,148
Germany,United States,1336,1468,132
United States,Germany,1468,1336,132
Netherlands,United States,660,776,116
United States,Netherlands,776,660,116


Databricks visualization. Run in Databricks to view.

In [0]:
%python
display(spark.sql("""
    SELECT
        CASE
            WHEN flight_count >= 1000 THEN 'High (1000+)'
            WHEN flight_count >= 100  THEN 'Medium (100-999)'
            ELSE 'Low (<100)'
        END AS volume_category,
        COUNT(*) AS route_count
    FROM flights_clean
    GROUP BY volume_category
    ORDER BY route_count DESC
"""))  # ← display() + GROUP BY for pie chart

volume_category,route_count
Medium (100-999),105
Low (<100),89
High (1000+),14


Databricks visualization. Run in Databricks to view.